# Notebook 4 – Entity linking, predicate alignment, and knowledge base expansion

In this notebook, we move from the **initial private RDF graph** to a more connected and semantically grounded knowledge base.

The goals of this notebook are:
1. Align film nodes and selected local entities with **Wikidata**.
2. Align local predicates with equivalent **Wikidata properties**.
3. Expand the graph using **one-hop Wikidata entity data**.

This notebook avoids direct dependence on the unstable SPARQL query service by using:
- **Wikidata EntityData JSON**
- **Wikidata API label resolution**

The outputs of this notebook will support the next project stages:
- reasoning,
- knowledge graph embeddings,
- RDF/SPARQL-based RAG.

The main outputs of this notebook will be:
- `/content/kg_artifacts/alignment_graph.ttl`
- `/content/kg_artifacts/expanded_graph.ttl`
- `/content/kg_artifacts/expanded_graph.nt`
- `/content/kg_artifacts/combined_expanded_graph.ttl`

In [1]:
# Cell 2 — Imports, paths, and load Notebook 3 artifacts

# Run once in Colab if needed
%pip install -q rdflib pandas tqdm

import os
import re
import json
import time
import random
import unicodedata
import requests
import pandas as pd

from tqdm.auto import tqdm
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL, XSD

# Paths
films_path = "/content/wikidata_films.json"
ontology_path = "/content/ontology.ttl"
initial_graph_path = "/content/initial_graph.ttl"
combined_graph_path = "/content/combined_graph.ttl"

for path in [films_path, ontology_path, initial_graph_path, combined_graph_path]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")

# Load films metadata
with open(films_path, "r", encoding="utf-8") as f:
    films = json.load(f)

films_df = pd.DataFrame(films)

# Load graphs
ontology_graph = Graph()
ontology_graph.parse(ontology_path, format="turtle")

initial_graph = Graph()
initial_graph.parse(initial_graph_path, format="turtle")

combined_graph = Graph()
combined_graph.parse(combined_graph_path, format="turtle")

print("=== Notebook 4 input summary ===")
print(f"Films metadata rows: {len(films_df)}")
print(f"Ontology triples: {len(ontology_graph)}")
print(f"Initial graph triples: {len(initial_graph)}")
print(f"Combined graph triples: {len(combined_graph)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 7.0 MB/s eta 0:00:00
=== Notebook 4 input summary ===
Films metadata rows: 998
Ontology triples: 77
Initial graph triples: 31562
Combined graph triples: 31639


In [2]:
# Cell 3 — Namespaces, helpers, and Wikidata API functions

EX = Namespace("http://example.org/movie#")
EXREL = Namespace("http://example.org/movie/relation/")
WD = Namespace("http://www.wikidata.org/entity/")
WDT = Namespace("http://www.wikidata.org/prop/direct/")

HEADERS = {
    "User-Agent": "MoviesKGBot/1.0 (university project; educational use)"
}

session = requests.Session()
session.headers.update(HEADERS)

WIKIDATA_API = "https://www.wikidata.org/w/api.php"

def slugify(value: str) -> str:
    if value is None:
        return "unknown"
    value = str(value).strip()
    value = unicodedata.normalize("NFKD", value).encode("ascii", "ignore").decode("ascii")
    value = value.lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    value = value.strip("_")
    return value or "unknown"

def make_film_uri(film_id: str) -> URIRef:
    return EX[f"film_{film_id}"]

def make_entity_uri(text: str) -> URIRef:
    return EX[f"entity_{slugify(text)}"]

def make_wikidata_proxy_uri(qid: str) -> URIRef:
    return EX[f"wd_{qid}"]

label_cache = {}

def fetch_entity_data(qid, max_retries=4, base_sleep=2):
    """
    Fetch EntityData JSON for one Wikidata entity.
    """
    url = f"https://www.wikidata.org/wiki/Special:EntityData/{qid}.json"

    for attempt in range(max_retries):
        try:
            response = session.get(url, timeout=30)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            if attempt < max_retries - 1:
                wait = base_sleep * (2 ** attempt) + random.uniform(0, 1)
                print(f"EntityData error for {qid}: {e}. Retrying in {wait:.1f}s...")
                time.sleep(wait)
            else:
                print(f"Failed to fetch EntityData for {qid}")
                return None

def resolve_entity_labels(entity_ids, batch_size=50, max_retries=4, base_sleep=2):
    """
    Resolve Wikidata QIDs to English labels using wbgetentities.
    """
    unresolved = [eid for eid in entity_ids if eid not in label_cache]

    for start in range(0, len(unresolved), batch_size):
        batch = unresolved[start:start + batch_size]
        params = {
            "action": "wbgetentities",
            "ids": "|".join(batch),
            "props": "labels",
            "languages": "en",
            "format": "json",
        }

        success = False
        for attempt in range(max_retries):
            try:
                response = session.get(WIKIDATA_API, params=params, timeout=30)
                response.raise_for_status()
                data = response.json()

                for eid, entity_data in data.get("entities", {}).items():
                    label = entity_data.get("labels", {}).get("en", {}).get("value", eid)
                    label_cache[eid] = label

                success = True
                break

            except requests.exceptions.RequestException as e:
                if attempt < max_retries - 1:
                    wait = base_sleep * (2 ** attempt) + random.uniform(0, 1)
                    print(f"Label resolution error for batch {start}: {e}. Retrying in {wait:.1f}s...")
                    time.sleep(wait)
                else:
                    print(f"Failed label resolution for batch starting at {start}")
                    for eid in batch:
                        label_cache.setdefault(eid, eid)

        if not success:
            for eid in batch:
                label_cache.setdefault(eid, eid)

    return {eid: label_cache.get(eid, eid) for eid in entity_ids}

print("Helpers ready.")

Helpers ready.


In [3]:
# Cell 4 — Directly align film nodes to Wikidata

alignment_graph = Graph()
alignment_graph.bind("ex", EX)
alignment_graph.bind("wd", WD)
alignment_graph.bind("owl", OWL)

qid_to_local_uri = {}
aligned_film_count = 0

for film in films:
    qid = film["id"]
    film_uri = make_film_uri(qid)
    wikidata_uri = WD[qid]

    alignment_graph.add((film_uri, OWL.sameAs, wikidata_uri))
    qid_to_local_uri[qid] = film_uri
    aligned_film_count += 1

print(f"Aligned film nodes to Wikidata: {aligned_film_count}")

Aligned film nodes to Wikidata: 998


In [4]:
# Cell 5 — Align local structured entities using batched Wikidata API calls

ALIGNMENT_PROPERTIES = {
    "P57": "directors",
    "P136": "genres",
    "P495": "countries",
    "P161": "cast",
    "P166": "awards",
    "P272": "production_companies",
    "P156": "followed_by",
    "P155": "preceded_by",
}

def fetch_entities_batch(qids, max_retries=4, base_sleep=3):
    """
    Fetch multiple Wikidata entities in one request using wbgetentities.
    """
    params = {
        "action": "wbgetentities",
        "ids": "|".join(qids),
        "props": "claims",
        "format": "json",
    }

    for attempt in range(max_retries):
        try:
            response = session.get(WIKIDATA_API, params=params, timeout=30)
            response.raise_for_status()
            return response.json().get("entities", {})

        except requests.exceptions.RequestException as e:
            if attempt < max_retries - 1:
                wait = base_sleep * (2 ** attempt) + random.uniform(0, 1)
                print(f"Batch entity fetch error: {e}. Retrying in {wait:.1f}s...")
                time.sleep(wait)
            else:
                print(f"Failed to fetch entity batch: {qids[:3]} ...")
                return {}

aligned_entity_count = 0
batch_size = 20  # reduce if you still get rate-limited

for start in tqdm(range(0, len(films), batch_size), desc="Aligning local entities"):
    batch_films = films[start:start + batch_size]
    batch_qids = [film["id"] for film in batch_films]

    entities_data = fetch_entities_batch(batch_qids)
    if not entities_data:
        time.sleep(1.5)
        continue

    # collect all linked QIDs we will need labels for
    linked_entity_ids = set()
    per_film_property_ids = {}

    for film in batch_films:
        qid = film["id"]
        claims = entities_data.get(qid, {}).get("claims", {})
        per_film_property_ids[qid] = {}

        for prop in ALIGNMENT_PROPERTIES.keys():
            per_film_property_ids[qid][prop] = []

            for claim in claims.get(prop, []):
                mainsnak = claim.get("mainsnak", {})
                datavalue = mainsnak.get("datavalue")

                if not datavalue:
                    continue

                value = datavalue.get("value")
                if isinstance(value, dict) and value.get("entity-type") == "item":
                    entity_id = value.get("id")
                    if entity_id:
                        per_film_property_ids[qid][prop].append(entity_id)
                        linked_entity_ids.add(entity_id)

    # resolve labels once per batch
    id_to_label = resolve_entity_labels(sorted(linked_entity_ids), batch_size=50, base_sleep=3)

    # add alignments
    for film in batch_films:
        qid = film["id"]

        for prop, entity_ids in per_film_property_ids[qid].items():
            for entity_id in entity_ids:
                label = id_to_label.get(entity_id, entity_id)
                local_uri = make_entity_uri(label)

                # only align if this local node exists in the graph
                if (local_uri, None, None) in combined_graph or (None, None, local_uri) in combined_graph:
                    alignment_graph.add((local_uri, OWL.sameAs, WD[entity_id]))
                    qid_to_local_uri[entity_id] = local_uri
                    aligned_entity_count += 1

    time.sleep(1.0)

print(f"Aligned structured local entities: {aligned_entity_count}")

Aligning local entities:   0%|          | 0/50 [00:00<?, ?it/s]

Aligned structured local entities: 5756


In [5]:
# Cell 6 — Align local predicates to Wikidata properties

predicate_alignment = {
    EX.directedBy: WDT.P57,
    EX.hasCastMember: WDT.P161,
    EX.hasGenre: WDT.P136,
    EX.hasCountry: WDT.P495,
    EX.wonAward: WDT.P166,
    EX.producedBy: WDT.P272,
    EX.followedBy: WDT.P156,
    EX.precededBy: WDT.P155,
    EX.imdbId: WDT.P345,
}

for local_pred, wikidata_pred in predicate_alignment.items():
    alignment_graph.add((local_pred, OWL.equivalentProperty, wikidata_pred))

predicate_alignment_df = pd.DataFrame([
    {"local_predicate": str(local_pred), "wikidata_property": str(wikidata_pred)}
    for local_pred, wikidata_pred in predicate_alignment.items()
])

print(f"Aligned predicates: {len(predicate_alignment_df)}")
display(predicate_alignment_df)

Aligned predicates: 9


,local_predicate,wikidata_property
0,http://example.org/movie#directedBy,http://www.wikidata.org/prop/direct/P57
1,http://example.org/movie#hasCastMember,http://www.wikidata.org/prop/direct/P161
2,http://example.org/movie#hasGenre,http://www.wikidata.org/prop/direct/P136
3,http://example.org/movie#hasCountry,http://www.wikidata.org/prop/direct/P495
4,http://example.org/movie#wonAward,http://www.wikidata.org/prop/direct/P166
5,http://example.org/movie#producedBy,http://www.wikidata.org/prop/direct/P272
6,http://example.org/movie#followedBy,http://www.wikidata.org/prop/direct/P156
7,http://example.org/movie#precededBy,http://www.wikidata.org/prop/direct/P155
8,http://example.org/movie#imdbId,http://www.wikidata.org/prop/direct/P345


In [6]:
# Cell 7 — Define one-hop expansion property maps

# Film-centric one-hop expansion
FILM_EXPANSION_PROPERTIES = {
    "P57": EX.directedBy,
    "P136": EX.hasGenre,
    "P495": EX.hasCountry,
    "P161": EX.hasCastMember,
    "P166": EX.wonAward,
    "P272": EX.producedBy,
    "P156": EX.followedBy,
    "P155": EX.precededBy,
    "P364": EX.hasOriginalLanguage,
    "P58": EX.writtenBy,
    "P162": EX.producedByPerson,
    "P86": EX.composedBy,
    "P344": EX.cinematographyBy,
}

# One-hop expansion for aligned non-film entities
ENTITY_EXPANSION_PROPERTIES = {
    "P31": EX.instanceOf,
    "P106": EX.occupation,
    "P27": EX.hasCitizenship,
    "P19": EX.placeOfBirth,
    "P17": EX.locatedInCountry,
    "P279": EX.subclassOf,
}

# Declare new local predicates if they were not in the original ontology
for predicate_uri in list(FILM_EXPANSION_PROPERTIES.values()) + list(ENTITY_EXPANSION_PROPERTIES.values()):
    combined_graph.add((predicate_uri, RDF.type, RDF.Property))
    combined_graph.add((predicate_uri, RDFS.label, Literal(predicate_uri.split("#")[-1])))

print("Expansion property maps ready.")

Expansion property maps ready.


In [7]:
# Cell 8 — Expand the KB from aligned Wikidata entities using batched API calls

expanded_graph = Graph()
expanded_graph.bind("ex", EX)
expanded_graph.bind("wd", WD)
expanded_graph.bind("owl", OWL)

def get_local_uri_for_qid(qid, label=None):
    """
    Reuse aligned local URIs if available; otherwise create local/proxy URIs.
    """
    if qid in qid_to_local_uri:
        return qid_to_local_uri[qid]

    if label:
        candidate_uri = make_entity_uri(label)
        qid_to_local_uri[qid] = candidate_uri
        return candidate_uri

    proxy_uri = make_wikidata_proxy_uri(qid)
    qid_to_local_uri[qid] = proxy_uri
    return proxy_uri

expanded_edge_count = 0

# ---------- Part A: Expand from film entities in batches ----------
film_batch_size = 20   # lower to 10 if you still hit 429

for start in tqdm(range(0, len(films), film_batch_size), desc="Expanding film entities"):
    batch_films = films[start:start + film_batch_size]
    batch_qids = [film["id"] for film in batch_films]

    entities_data = fetch_entities_batch(batch_qids, base_sleep=3)
    if not entities_data:
        time.sleep(2.0)
        continue

    linked_entity_ids = set()
    per_film_property_ids = {}

    for film in batch_films:
        qid = film["id"]
        claims = entities_data.get(qid, {}).get("claims", {})
        per_film_property_ids[qid] = {}

        for prop in FILM_EXPANSION_PROPERTIES.keys():
            per_film_property_ids[qid][prop] = []

            for claim in claims.get(prop, []):
                mainsnak = claim.get("mainsnak", {})
                datavalue = mainsnak.get("datavalue")

                if not datavalue:
                    continue

                value = datavalue.get("value")
                if isinstance(value, dict) and value.get("entity-type") == "item":
                    entity_id = value.get("id")
                    if entity_id:
                        per_film_property_ids[qid][prop].append(entity_id)
                        linked_entity_ids.add(entity_id)

    id_to_label = resolve_entity_labels(
        sorted(linked_entity_ids),
        batch_size=50,
        base_sleep=3
    )

    for film in batch_films:
        qid = film["id"]
        local_subject = make_film_uri(qid)

        for prop, entity_ids in per_film_property_ids[qid].items():
            predicate_uri = FILM_EXPANSION_PROPERTIES[prop]

            for entity_id in entity_ids:
                label = id_to_label.get(entity_id, entity_id)
                local_object = get_local_uri_for_qid(entity_id, label)

                expanded_graph.add((local_subject, predicate_uri, local_object))
                expanded_graph.add((local_object, RDFS.label, Literal(label)))
                expanded_graph.add((local_object, OWL.sameAs, WD[entity_id]))
                expanded_edge_count += 1

    time.sleep(1.5)

# ---------- Part B: Expand from aligned non-film entities in batches ----------
aligned_non_film_qids = [
    qid for qid in qid_to_local_uri.keys()
    if qid not in {film["id"] for film in films}
]

# keep this cap for runtime safety
aligned_non_film_qids = aligned_non_film_qids[:300]

entity_batch_size = 20   # lower to 10 if needed

for start in tqdm(range(0, len(aligned_non_film_qids), entity_batch_size), desc="Expanding linked entities"):
    batch_qids = aligned_non_film_qids[start:start + entity_batch_size]

    entities_data = fetch_entities_batch(batch_qids, base_sleep=3)
    if not entities_data:
        time.sleep(2.0)
        continue

    linked_entity_ids = set()
    per_entity_property_ids = {}

    for qid in batch_qids:
        claims = entities_data.get(qid, {}).get("claims", {})
        per_entity_property_ids[qid] = {}

        for prop in ENTITY_EXPANSION_PROPERTIES.keys():
            per_entity_property_ids[qid][prop] = []

            for claim in claims.get(prop, []):
                mainsnak = claim.get("mainsnak", {})
                datavalue = mainsnak.get("datavalue")

                if not datavalue:
                    continue

                value = datavalue.get("value")
                if isinstance(value, dict) and value.get("entity-type") == "item":
                    entity_id = value.get("id")
                    if entity_id:
                        per_entity_property_ids[qid][prop].append(entity_id)
                        linked_entity_ids.add(entity_id)

    id_to_label = resolve_entity_labels(
        sorted(linked_entity_ids),
        batch_size=50,
        base_sleep=3
    )

    for qid in batch_qids:
        local_subject = qid_to_local_uri[qid]

        for prop, entity_ids in per_entity_property_ids[qid].items():
            predicate_uri = ENTITY_EXPANSION_PROPERTIES[prop]

            for entity_id in entity_ids:
                label = id_to_label.get(entity_id, entity_id)
                local_object = get_local_uri_for_qid(entity_id, label)

                expanded_graph.add((local_subject, predicate_uri, local_object))
                expanded_graph.add((local_object, RDFS.label, Literal(label)))
                expanded_graph.add((local_object, OWL.sameAs, WD[entity_id]))
                expanded_edge_count += 1

    time.sleep(1.5)

combined_expanded_graph = combined_graph + alignment_graph + expanded_graph

print(f"Expansion complete. Added {expanded_edge_count} edges.")
print(f"Expanded graph triples: {len(expanded_graph)}")
print(f"Combined expanded graph triples: {len(combined_expanded_graph)}")

Expanding film entities:   0%|          | 0/50 [00:00<?, ?it/s]

Expanding linked entities:   0%|          | 0/15 [00:00<?, ?it/s]

Expansion complete. Added 9400 edges.
Expanded graph triples: 19875
Combined expanded graph triples: 42958


In [8]:
# Cell 9 — Save alignment and expanded graph artifacts

import os

output_dir = "/content/kg_artifacts"
os.makedirs(output_dir, exist_ok=True)

alignment_path = os.path.join(output_dir, "alignment_graph.ttl")
expanded_graph_ttl_path = os.path.join(output_dir, "expanded_graph.ttl")
expanded_graph_nt_path = os.path.join(output_dir, "expanded_graph.nt")
combined_expanded_path = os.path.join(output_dir, "combined_expanded_graph.ttl")

alignment_graph.serialize(destination=alignment_path, format="turtle")
expanded_graph.serialize(destination=expanded_graph_ttl_path, format="turtle")
expanded_graph.serialize(destination=expanded_graph_nt_path, format="nt")
combined_expanded_graph.serialize(destination=combined_expanded_path, format="turtle")

print("Saved alignment graph to:", alignment_path)
print("Saved expanded graph (TTL) to:", expanded_graph_ttl_path)
print("Saved expanded graph (NT) to:", expanded_graph_nt_path)
print("Saved combined expanded graph to:", combined_expanded_path)

/usr/local/lib/python3.12/dist-packages/rdflib/plugins/serializers/nt.py:39: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


Saved alignment graph to: /content/kg_artifacts/alignment_graph.ttl
Saved expanded graph (TTL) to: /content/kg_artifacts/expanded_graph.ttl
Saved expanded graph (NT) to: /content/kg_artifacts/expanded_graph.nt
Saved combined expanded graph to: /content/kg_artifacts/combined_expanded_graph.ttl


In [9]:
# Cell 10 — Alignment and expansion statistics

def graph_stats(graph: Graph):
    subjects = set()
    objects = set()
    predicates = set()

    for s, p, o in graph:
        if isinstance(s, URIRef):
            subjects.add(s)
        if isinstance(o, URIRef):
            objects.add(o)
        predicates.add(p)

    entity_nodes = subjects.union(objects)

    return {
        "triples": len(graph),
        "unique_entities": len(entity_nodes),
        "unique_predicates": len(predicates),
    }

alignment_stats = graph_stats(alignment_graph)
expanded_stats = graph_stats(expanded_graph)
combined_expanded_stats = graph_stats(combined_expanded_graph)

print("=== Alignment graph statistics ===")
for k, v in alignment_stats.items():
    print(f"{k}: {v}")

print("\n=== Expanded graph statistics ===")
for k, v in expanded_stats.items():
    print(f"{k}: {v}")

print("\n=== Combined expanded graph statistics ===")
for k, v in combined_expanded_stats.items():
    print(f"{k}: {v}")

# SameAs count
sameas_count = sum(1 for _ in alignment_graph.triples((None, OWL.sameAs, None)))
equivprop_count = sum(1 for _ in alignment_graph.triples((None, OWL.equivalentProperty, None)))

print("\n=== Alignment QA ===")
print(f"owl:sameAs links: {sameas_count}")
print(f"owl:equivalentProperty links: {equivprop_count}")

# Sample expanded triples
sample_expanded_rows = []
for i, (s, p, o) in enumerate(expanded_graph):
    sample_expanded_rows.append({
        "subject": str(s),
        "predicate": str(p),
        "object": str(o),
    })
    if i >= 19:
        break

print("\n=== Sample expanded triples ===")
display(pd.DataFrame(sample_expanded_rows))

=== Alignment graph statistics ===
triples: 4850
unique_entities: 9698
unique_predicates: 2

=== Expanded graph statistics ===
triples: 19875
unique_entities: 11344
unique_predicates: 21

=== Combined expanded graph statistics ===
triples: 42958
unique_entities: 15872
unique_predicates: 127

=== Alignment QA ===
owl:sameAs links: 4841
owl:equivalentProperty links: 9

=== Sample expanded triples ===


,subject,predicate,object
0,http://example.org/movie#entity_gothic_horror_...,http://www.w3.org/2000/01/rdf-schema#label,gothic horror film
1,http://example.org/movie#entity_soledad_villamil,http://www.w3.org/2000/01/rdf-schema#label,Soledad Villamil
2,http://example.org/movie#entity_sohaila_kapur,http://www.w3.org/2000/01/rdf-schema#label,Sohaila Kapur
3,http://example.org/movie#entity_aparna_balamurali,http://example.org/movie#instanceOf,http://example.org/movie#entity_human
4,http://example.org/movie#entity_myriam_leblanc,http://www.w3.org/2000/01/rdf-schema#label,Myriam LeBlanc
5,http://example.org/movie#entity_anees_bazmee,http://www.w3.org/2002/07/owl#sameAs,http://www.wikidata.org/entity/Q652149
6,http://example.org/movie#entity_john_farrelly,http://www.w3.org/2000/01/rdf-schema#label,John Farrelly
7,http://example.org/movie#entity_greg_hsu,http://example.org/movie#occupation,http://example.org/movie#entity_model
8,http://example.org/movie#entity_mylene_mackay,http://www.w3.org/2000/01/rdf-schema#label,Mylène Mackay
9,http://example.org/movie#entity_milla_jovovich,http://www.w3.org/2000/01/rdf-schema#label,Milla Jovovich


## Notebook 4 summary

This notebook transformed the original local RDF graph into a **linked and enriched knowledge graph** by aligning local movie entities with Wikidata and then expanding the graph with additional one-hop external knowledge.

### What was done

The first stage focused on **entity and predicate alignment**.  
Local film nodes were linked directly to Wikidata using `owl:sameAs`, and several categories of structured local entities were also aligned whenever reliable matches were available. These included:
- directors,
- genres,
- countries,
- cast members,
- awards,
- and companies.

In parallel, the main local predicates were aligned to external semantics using `owl:equivalentProperty`.

The second stage focused on **graph expansion**.  
Starting from the aligned entities, the notebook retrieved one-hop Wikidata facts and added them to the project graph. This introduced new semantic relations such as:
- `instanceOf`,
- `occupation`,
- `hasCitizenship`,
- `placeOfBirth`,
- `writtenBy`,
- and other movie-related links connected to people, places, organizations, and types.

### Main results

The alignment step created an explicit semantic bridge between the local graph and Wikidata.

#### Alignment graph statistics
- **Triples:** 4850
- **Unique entities:** 9698
- **Unique predicates:** 2
- **`owl:sameAs` links:** 4841
- **`owl:equivalentProperty` links:** 9

These results show that a large number of local nodes are now explicitly linked to Wikidata, which gives the graph a much stronger semantic foundation for downstream tasks.

#### Expanded graph statistics
- **Triples:** 19875
- **Unique entities:** 11344
- **Unique predicates:** 21

The expansion stage substantially increased the size and semantic richness of the graph by adding one-hop external knowledge around already aligned entities.

#### Final combined graph statistics
After merging the ontology, the initial graph, the alignment graph, and the expanded graph, the final combined graph reached:
- **Triples:** 42958
- **Unique entities:** 15872
- **Unique predicates:** 127

Compared with the initial graph, this is a major increase in both connectivity and coverage.

### What the enriched graph now contains

The expanded graph is no longer limited to basic movie metadata.  
It now includes:
- explicit links from local entities to Wikidata,
- labels for aligned and expanded entities,
- person-level facts such as occupations and citizenship,
- place-level facts such as birth locations,
- additional creative-role relations such as writers and producers,
- and broader neighborhood information around films, cast members, directors, awards, and related entities.

In practice, this means the graph evolved from a relatively local RDF dataset into a much richer **linked knowledge graph** with external grounding and broader semantic context.

### Main outputs

This notebook produced the following artifacts:
- `/content/kg_artifacts/alignment_graph.ttl`
- `/content/kg_artifacts/expanded_graph.ttl`
- `/content/kg_artifacts/expanded_graph.nt`
- `/content/kg_artifacts/combined_expanded_graph.ttl`

### Main findings

This notebook significantly improved the project knowledge base.  
The graph is now:
- larger,
- denser,
- externally grounded,
- and more useful for downstream tasks.

The alignment step improved interoperability with Wikidata, while the expansion step introduced richer contextual knowledge around films and related entities. This provides a much better base for:
- rule-based reasoning,
- knowledge graph embedding,
- and RDF/SPARQL-based graph-grounded question answering.

### Why this matters

This notebook is a key transition point in the project.  
It is where the private RDF graph becomes a **linked and enriched knowledge graph**.

That matters because downstream methods depend strongly on graph quality and connectivity. By aligning local entities to Wikidata and expanding their neighborhoods, the graph becomes much better suited for:
- ontology-based reasoning,
- knowledge graph embedding,
- and RDF/SPARQL-based retrieval and question answering.

### Next step

The next notebook focuses on the **reasoning and KGE stage** of the project.  
It uses the enriched graph for:
1. rule-based reasoning,
2. knowledge graph embedding,
3. and preparation for graph-grounded QA in the final notebook.